In [ ]:
import netCDF4 as nc
import numpy as np
import os
import shutil

# =========================================================================
# SCRIPT: generate_wind_IC_ready.py
# PURPOSE: 
# 1. Reads wind_IC_19_22_flipped.nc
# 2. Renames 'valid_time' -> 'time'
# 3. Sets _FillValue during creation (fixes NC_ELATEFILL error)
# =========================================================================

# --- Configuration ---
input_file = '/g/data/ps29/nd0349/standalone/ww3/WW3/build_om3/input/1968_era5_winds_flipped.nc'
output_file = '/g/data/ps29/nd0349/standalone/ww3/WW3/build_om3/input/1968_era5_winds_ready.nc'

# vars_to_copy = ['u10', 'v10', 'siconc']
vars_to_copy = ['u10', 'v10']

# Clean up previous failed attempts
if os.path.isfile(output_file):
    os.remove(output_file)

print(f"Processing: {input_file} -> {output_file}")

try:
    # --- 1. READ INPUT DATA ---
    src = nc.Dataset(input_file, 'r')
    
    lon_data = src.variables['longitude'][:]
    lat_data = src.variables['latitude'][:]
    time_data = src.variables['time'][:]

    # --- 2. CREATE OUTPUT FILE ---
    dst = nc.Dataset(output_file, 'w', format='NETCDF4')

    # Dimensions
    dst.createDimension('longitude', len(lon_data))
    dst.createDimension('latitude', len(lat_data))
    dst.createDimension('time', len(time_data))

    # Variables
    lon_var = dst.createVariable('longitude', lon_data.dtype, ('longitude',))
    lat_var = dst.createVariable('latitude', lat_data.dtype, ('latitude',))
    time_var = dst.createVariable('time', time_data.dtype, ('time',))

    lon_var[:] = lon_data
    lat_var[:] = lat_data
    time_var[:] = time_data

    # Copy attributes
    for attr_name in src.variables['longitude'].ncattrs():
        lon_var.setncattr(attr_name, src.variables['longitude'].getncattr(attr_name))
    for attr_name in src.variables['latitude'].ncattrs():
        lat_var.setncattr(attr_name, src.variables['latitude'].getncattr(attr_name))
    for attr_name in src.variables['time'].ncattrs():
        if attr_name not in ['_FillValue']:
            time_var.setncattr(attr_name, src.variables['time'].getncattr(attr_name))
    # Standard WW3 names
    time_var.setncattr('standard_name', 'time')
    time_var.setncattr('long_name', 'time')

    # print("Dimensions created. Renamed 'valid_time' -> 'time'.")

    # --- 3. PROCESS VARIABLES ---
    for var_name in vars_to_copy:
        print(f"Processing variable: {var_name} ... ", end='')

        var_in = src.variables[var_name]
        data = var_in[:]

        # Get FillValue if exists
        fill_value = getattr(var_in, '_FillValue', None)

        # Create variable in output with FillValue
        if fill_value is not None:
            var_out = dst.createVariable(var_name, var_in.dtype, ('longitude', 'latitude', 'time'), fill_value=fill_value)
        else:
            var_out = dst.createVariable(var_name, var_in.dtype, ('longitude', 'latitude', 'time'))

        var_out[:] = data.transpose()

        # Copy attributes except _FillValue
        for attr_name in var_in.ncattrs():
            if attr_name != '_FillValue':
                var_out.setncattr(attr_name, var_in.getncattr(attr_name))

        print("Done.")

    # Close files
    src.close()
    dst.close()

    print("\n------------------------------------------------------")
    print(f"SUCCESS! File '{output_file}' is ready.")
    print("------------------------------------------------------")

except Exception as e:
    print(f"\nERROR: {e}")